In [1]:
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number, coalesce, lit, when, regexp_extract, split, sum, avg, first, length, substring, count, to_date
import matplotlib.pyplot as plt
import pandas as pd
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import year, weekofyear, lpad, month
import re
from pyspark.sql.functions import split, col, size
from pyspark.sql.functions import col, coalesce, lit
from pyspark.sql.functions import col, sum

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 3, Finished, Available, Finished, False)

In [2]:
ericsson_fdd_daily_hourly = 'abfss://1114beb9-0a42-4fa1-9a4f-3459bdeded56@onelake.dfs.fabric.microsoft.com/6a948289-89b4-4d32-85fe-fedcd6c9aa9c/Tables/pm_data_hourly/ericsson_fdd_daily_hourly'
ericsson_l2600_daily_hourly = 'abfss://1114beb9-0a42-4fa1-9a4f-3459bdeded56@onelake.dfs.fabric.microsoft.com/6a948289-89b4-4d32-85fe-fedcd6c9aa9c/Tables/pm_data_hourly/ericsson_l2600_daily_hourly'
huawei_fdd_daily_hourly = 'abfss://1114beb9-0a42-4fa1-9a4f-3459bdeded56@onelake.dfs.fabric.microsoft.com/6a948289-89b4-4d32-85fe-fedcd6c9aa9c/Tables/pm_data_hourly/huawei_fdd_daily_hourly'

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 4, Finished, Available, Finished, False)

In [3]:
FROM_DATE = '2026-03-02'
TO_DATE = '2026-03-05'

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 5, Finished, Available, Finished, False)

In [4]:
def load_and_concat_tables(table_path_1, table_path_2):

    columns_to_select = [
        "date",
        "hour",
        "cell_name",
        "avg_connected_users",
        "dl_traffic_volume_gb",
        "avg_dl_user_thp_qci_8_kbps_num",
        "avg_dl_user_thp_qci_8_kbps_deno",
        "avg_dl_user_thp_qci_9_kbps_num",
        "avg_dl_user_thp_qci_9_kbps_deno",
        "sector"
    ]
    
    df1 = spark.read.format("delta").load(table_path_1) \
        .select(*columns_to_select) \
        .filter((col("date") >= FROM_DATE)&(col("date") <= TO_DATE))
    
    df2 = spark.read.format("delta").load(table_path_2) \
        .select(*columns_to_select) \
        .filter((col("date") >= FROM_DATE)&(col("date") <= TO_DATE))
    
    df = df1.union(df2)

    # Fill nulls with 0 for specified columns
    df = df.fillna(0, subset=[
        "avg_connected_users",
        "dl_traffic_volume_gb",
        "avg_dl_user_thp_qci_8_kbps_num",
        "avg_dl_user_thp_qci_8_kbps_deno",
        "avg_dl_user_thp_qci_9_kbps_num",
        "avg_dl_user_thp_qci_9_kbps_deno"
    ])

    return df


StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 6, Finished, Available, Finished, False)

In [5]:
def dataframe_dump(df, filename, keep_last_n=5):
    """
    Write DataFrame to another workspace's lakehouse using ABFS path
    Keep only the last N files and delete older ones
    Handles year breaks correctly (e.g., 2025 Week 52 < 2026 Week 1)
   
    Parameters:
    df: DataFrame to write
    filename: Base filename (without year/week suffix)
    keep_last_n: Number of most recent files to keep (default: 5)
    """
   
    # ABFS path for target lakehouse
    abfs_path = "abfss://1114beb9-0a42-4fa1-9a4f-3459bdeded56@onelake.dfs.fabric.microsoft.com/6a948289-89b4-4d32-85fe-fedcd6c9aa9c/Files/csv_dumps"
    output_folder = f"{abfs_path}"
   
    try:
        from notebookutils import mssparkutils

        new_filename = f"{filename}.csv"
        csv_filename = f"{output_folder}/{new_filename}"
       
        # Get list of existing files
        try:
            files_info = mssparkutils.fs.ls(output_folder)
            files = [f.name for f in files_info]
            print(f"Found {len(files)} files in directory")
        except Exception as e:
            print(f"⚠️ Could not list files: {str(e)}")
            files = []
       
        # Filter files matching the pattern: filename_YYYY_WW.csv
        pattern = re.compile(rf"^{re.escape(filename)}_(\d{{4}})_(\d{{2}})\.csv$")
        matching_files = []
       
        for file in files:
            match = pattern.match(file)
            if match:
                year = int(match.group(1))
                week = int(match.group(2))
                sort_key = year * 100 + week
                matching_files.append((file, year, week, sort_key))
       
        print(f"Found {len(matching_files)} matching files")
       
        # Sort by sort_key (newest first)
        matching_files.sort(key=lambda x: x[3], reverse=True)
       
        # Delete files beyond keep_last_n (excluding the one we're about to create)
        matching_files_filtered = [f for f in matching_files if f[0] != new_filename]
       
        if len(matching_files_filtered) >= keep_last_n:
            files_to_delete = matching_files_filtered[keep_last_n-1:]
           
            print(f"Will delete {len(files_to_delete)} old files")
           
            for file_to_delete, year, week, _ in files_to_delete:
                try:
                    file_path = f"{output_folder}/{file_to_delete}"
                    mssparkutils.fs.rm(file_path, False)
                    print(f"🗑️ Deleted old file: {file_to_delete} (Year: {year}, Week: {week})")
                except Exception as del_err:
                    print(f"⚠️ Could not delete {file_to_delete}: {str(del_err)}")
        else:
            print(f"Only {len(matching_files_filtered)} files exist, keeping all (threshold: {keep_last_n})")
       
        # Save the new file
        if hasattr(df, 'write'):
            # Spark DataFrame — write CSV directly to avoid toPandas() memory issues
            temp_path = f"{output_folder}/_temp_{filename}"
            df.coalesce(1).write.mode("overwrite").option("header", "true").csv(temp_path)
            
            # Find the part file and move it to final location
            temp_files = mssparkutils.fs.ls(temp_path)
            part_file = [f for f in temp_files if f.name.startswith("part-")][0]
            mssparkutils.fs.cp(f"{temp_path}/{part_file.name}", csv_filename, True)
            mssparkutils.fs.rm(temp_path, True)
            
            row_count = df.count()
        else:
            # Pandas DataFrame
            df.to_csv(csv_filename, index=False)
            row_count = len(df)
       
        print(f"✓ Exported: {csv_filename} - Rows: {row_count}")
        print(f"📁 Keeping last {keep_last_n} files")
       
    except Exception as e:
        print(f"✗ Failed to export {filename}: {str(e)}")

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 7, Finished, Available, Finished, False)

In [6]:
dfe = load_and_concat_tables(ericsson_fdd_daily_hourly, ericsson_l2600_daily_hourly)

dfh = spark.read.format("delta").load(huawei_fdd_daily_hourly) \
    .select(
        to_date(col("time")).alias("date"),
        col("hour_key").alias("hour"),
        col("cell_name").alias("cell_name"),
        col("rrc_user_num").alias("avg_connected_users"),
        col("dl_traffic_volume_gb_gb").alias("dl_traffic_volume_gb"),
        (col("l_thrp_bits_dl_qci_8_bit") - col("l_thrp_bits_dl_lasttti_qci_8_bit")).alias("avg_dl_user_thp_qci_8_kbps_num"),
        (col("l_thrp_bits_dl_qci_9_bit") - col("l_thrp_bits_dl_lasttti_qci_9_bit")).alias("avg_dl_user_thp_qci_9_kbps_num"),
        col("l_thrp_time_dl_rmvlasttti_qci_8_ms").alias("avg_dl_user_thp_qci_8_kbps_deno"),
        col("l_thrp_time_dl_rmvlasttti_qci_9_ms").alias("avg_dl_user_thp_qci_9_kbps_deno"),
        col("sector").alias("sector")
    ) \
    .filter(col("date") >= FROM_DATE)

df_combined = dfe.unionByName(dfh)

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 8, Finished, Available, Finished, False)

In [7]:
df_combined.head

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 9, Finished, Available, Finished, False)

<bound method DataFrame.head of DataFrame[date: timestamp, hour: string, cell_name: string, avg_connected_users: string, dl_traffic_volume_gb: string, avg_dl_user_thp_qci_8_kbps_num: double, avg_dl_user_thp_qci_8_kbps_deno: string, avg_dl_user_thp_qci_9_kbps_num: double, avg_dl_user_thp_qci_9_kbps_deno: string, sector: string]>

In [8]:
from pyspark.sql.functions import split, col, when, size

df_combined = df_combined.withColumn(
    "site_name",
    when(size(split(col("cell_name"), "-")) > 1,
         split(col("cell_name"), "-")[1]
    ).otherwise(None)
)

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 10, Finished, Available, Finished, False)

In [9]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, DoubleType

df_combined = df_combined.withColumn("hour", col("hour").cast(IntegerType())) \
       .withColumn("avg_connected_users", col("avg_connected_users").cast(DoubleType())) \
       .withColumn("dl_traffic_volume_gb", col("dl_traffic_volume_gb").cast(DoubleType())) \
       .withColumn("avg_dl_user_thp_qci_8_kbps_deno", col("avg_dl_user_thp_qci_8_kbps_deno").cast(DoubleType())) \
       .withColumn("avg_dl_user_thp_qci_9_kbps_deno", col("avg_dl_user_thp_qci_9_kbps_deno").cast(DoubleType()))

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 11, Finished, Available, Finished, False)

In [10]:
from pyspark.sql.functions import sum

# Aggregate all numeric columns by date(timestamp), hour, and site_name
df_site = df_combined.groupBy("date", "hour", "site_name").agg(
    
    sum("avg_connected_users").alias("sum_avg_connected_users"),
    
    sum("dl_traffic_volume_gb").alias("sum_dl_traffic_volume_gb"),
    
    sum("avg_dl_user_thp_qci_8_kbps_num").alias("sum_avg_dl_user_thp_qci_8_kbps_num"),
    
    sum("avg_dl_user_thp_qci_8_kbps_deno").alias("sum_avg_dl_user_thp_qci_8_kbps_deno"),
    
    sum("avg_dl_user_thp_qci_9_kbps_num").alias("sum_avg_dl_user_thp_qci_9_kbps_num"),
    
    sum("avg_dl_user_thp_qci_9_kbps_deno").alias("sum_avg_dl_user_thp_qci_9_kbps_deno")
)

# Show results
df_site.show(truncate=False)

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 12, Finished, Available, Finished, False)

+-------------------+----+---------+-----------------------+------------------------+----------------------------------+-----------------------------------+----------------------------------+-----------------------------------+
|date               |hour|site_name|sum_avg_connected_users|sum_dl_traffic_volume_gb|sum_avg_dl_user_thp_qci_8_kbps_num|sum_avg_dl_user_thp_qci_8_kbps_deno|sum_avg_dl_user_thp_qci_9_kbps_num|sum_avg_dl_user_thp_qci_9_kbps_deno|
+-------------------+----+---------+-----------------------+------------------------+----------------------------------+-----------------------------------+----------------------------------+-----------------------------------+
|2026-04-28 00:00:00|19  |GM0131   |1074.3399000000002     |89.83109999999999       |3.1191093184E11                   |2.02393257E8                       |2.98789252672E11                  |3.56576742E8                       |
|2026-03-22 00:00:00|9   |GM0006   |1512.0549999999998     |123.73940000000002      |4.5

In [21]:
#daily per site
from pyspark.sql.functions import sum, first

# Aggregate all numeric columns by date(timestamp) and site_name
df_site = df_combined.groupBy("date", "site_name").agg(
    
    # If hour is same within a timestamp, take first value
    first("hour").alias("hour"),
    
    sum("avg_connected_users").alias("sum_avg_connected_users"),
    
    sum("dl_traffic_volume_gb").alias("sum_dl_traffic_volume_gb"),
    
    sum("avg_dl_user_thp_qci_8_kbps_num").alias("sum_avg_dl_user_thp_qci_8_kbps_num"),
    
    sum("avg_dl_user_thp_qci_8_kbps_deno").alias("sum_avg_dl_user_thp_qci_8_kbps_deno"),
    
    sum("avg_dl_user_thp_qci_9_kbps_num").alias("sum_avg_dl_user_thp_qci_9_kbps_num"),
    
    sum("avg_dl_user_thp_qci_9_kbps_deno").alias("sum_avg_dl_user_thp_qci_9_kbps_deno")
)

# Show results
df_site.show(truncate=False)

StatementMeta(, 91ad306b-8394-4d6c-b031-733b536a7cf7, 28, Finished, Available, Finished, False)

+-------------------+---------+----+-----------------------+------------------------+----------------------------------+-----------------------------------+----------------------------------+-----------------------------------+
|date               |site_name|hour|sum_avg_connected_users|sum_dl_traffic_volume_gb|sum_avg_dl_user_thp_qci_8_kbps_num|sum_avg_dl_user_thp_qci_8_kbps_deno|sum_avg_dl_user_thp_qci_9_kbps_num|sum_avg_dl_user_thp_qci_9_kbps_deno|
+-------------------+---------+----+-----------------------+------------------------+----------------------------------+-----------------------------------+----------------------------------+-----------------------------------+
|2026-03-16 00:00:00|GM5191   |14  |8183.460899999999      |786.8856                |3.149926709736E12                 |3.74146395E8                       |1.528820434352E12                 |6.42464244E8                       |
|2026-03-16 00:00:00|CM1667   |15  |8993.9697              |809.1816                |2.7

In [11]:
df_site.head

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 13, Finished, Available, Finished, False)

<bound method DataFrame.head of DataFrame[date: timestamp, hour: int, site_name: string, sum_avg_connected_users: double, sum_dl_traffic_volume_gb: double, sum_avg_dl_user_thp_qci_8_kbps_num: double, sum_avg_dl_user_thp_qci_8_kbps_deno: double, sum_avg_dl_user_thp_qci_9_kbps_num: double, sum_avg_dl_user_thp_qci_9_kbps_deno: double]>

In [18]:
display(df_site)

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 41cce228-7696-446b-a7e4-9254d0d798ee)

In [19]:
# Cache the final result before writing
df_site = df_site.cache()
df_site.count()

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 21, Finished, Available, Finished, False)

6577739

In [20]:
## Export to csv
dataframes_to_export = [
    (df_site, "4G_daily_KPI_Summary_Site_Hourly_March_2-5_new"),
]
 
# Run all exports
for df_site, filename in dataframes_to_export:
    dataframe_dump(df_site, filename, 20)

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 22, Finished, Available, Finished, False)

Found 18 files in directory
Found 0 matching files
Only 0 files exist, keeping all (threshold: 20)
✓ Exported: abfss://1114beb9-0a42-4fa1-9a4f-3459bdeded56@onelake.dfs.fabric.microsoft.com/6a948289-89b4-4d32-85fe-fedcd6c9aa9c/Files/csv_dumps/4G_daily_KPI_Summary_Site_Hourly_March_2-5_new.csv - Rows: 6577739
📁 Keeping last 20 files


StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 23, Finished, Available, Finished, False)

Py4JJavaError: An error occurred while calling o8397.csv.
: Operation failed: "Bad Request", 400, HEAD, http://onelake.dfs.fabric.microsoft.com/1114beb9-0a42-4fa1-9a4f-3459bdeded56/6a948289-89b4-4d32-85fe-fedcd6c9aa9c/4G_daily_KPI_Summary_Site_Hourly_March_2-5_new?upn=false&action=getStatus&timeout=90
	at org.apache.hadoop.fs.azurebfs.services.AbfsRestOperation.completeExecute(AbfsRestOperation.java:231)
	at org.apache.hadoop.fs.azurebfs.services.AbfsRestOperation.lambda$execute$0(AbfsRestOperation.java:191)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDurationOfInvocation(IOStatisticsBinding.java:464)
	at org.apache.hadoop.fs.azurebfs.services.AbfsRestOperation.execute(AbfsRestOperation.java:189)
	at org.apache.hadoop.fs.azurebfs.services.AbfsClient.getPathStatus(AbfsClient.java:779)
	at org.apache.hadoop.fs.azurebfs.AzureBlobFileSystemStore.getFileStatus(AzureBlobFileSystemStore.java:1089)
	at org.apache.hadoop.fs.azurebfs.AzureBlobFileSystem.getFileStatus(AzureB

StatementMeta(, 0f4df17e-fae4-4f60-82b8-f1a0ab75db44, 26, Finished, Available, Finished, False)

In [22]:
df_site.to_csv("Files/csv_dumps/4G_daily_KPI_Summary_Hourly_March_2-5_new_sites", index=False)

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 24, Finished, Available, Finished, False)

AttributeError: 'DataFrame' object has no attribute 'to_csv'

In [23]:
## Export to csv
dataframes_to_export = [
    (df_site, "4G_daily_KPI_Summary_Site_Hourly_March_2-5_new"),
]
 
# Run all exports
for df_site, filename in dataframes_to_export:
    dataframe_dump(df_site, filename, 20)

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 25, Finished, Available, Finished, False)

Found 18 files in directory
Found 0 matching files
Only 0 files exist, keeping all (threshold: 20)
✓ Exported: abfss://1114beb9-0a42-4fa1-9a4f-3459bdeded56@onelake.dfs.fabric.microsoft.com/6a948289-89b4-4d32-85fe-fedcd6c9aa9c/Files/csv_dumps/4G_daily_KPI_Summary_Site_Hourly_March_2-5_new.csv - Rows: 6577739
📁 Keeping last 20 files


In [24]:
df_site.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("Files/csv_dumps/4G_daily_KPI_Summary_Hourly_March_2-5_new_sites")

StatementMeta(, 2263134e-13da-493a-8ba4-032c7e043496, 26, Finished, Available, Finished, False)